In [3]:
import torch


def vector_rms_norm(z, zero_mean=False, eps=1e-6):
    assert z.ndim in [3, 4]  # [B, C, H, W] or [B, N, D]
    dim = tuple(range(1, z.ndim))  # w/o batch dimension
    if zero_mean:
        z = z - z.mean(dim=dim, keepdim=True)
    m = z.square().mean(dim=dim, keepdim=True)
    m = torch.rsqrt(m + eps).to(z.dtype)
    return z * m


def analyze(file_path, name="DATA"):
    data = torch.load(file_path)

    z = data["encodings"]
    y = data.get("labels")
    splits = data.get("split_ids")

    print(f"\n==== {name} ====")
    print("Shape:", z.shape)
    print("Dtype:", z.dtype)
    print("Min:", z.min().item())
    print("Max:", z.max().item())
    print("Mean:", z.mean().item())
    print("Std:", z.std().item())

    norms = torch.norm(z, dim=[1, 2])
    print("Norm mean:", norms.mean().item())
    print("Norm std:", norms.std().item())

    if y is not None:
        print("Labels:", y.shape, "unique:", torch.unique(y))

    if splits is not None:
        print("Splits:", splits.shape)
        for i in torch.unique(splits):
            print(f"  split {i.item()} count:", (splits == i).sum().item())


analyze(
    "/home/nub/UvA/Master/DL2/spherical-flow-matching/sphere-encoder-main/workspace/experiments/sphere-small-small-cifar-10-32px/encoding/encoded_dataset.pt"
)

/tmp/ipykernel_24953/91187656.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path)



==== DATA ====
Shape: torch.Size([60000, 256, 4])
Dtype: torch.bfloat16
Min: -6.75
Max: 7.125
Mean: 0.00086212158203125
Std: 1.0
Norm mean: 32.0
Norm std: 0.04052734375
Labels: torch.Size([60000]) unique: tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
Splits: torch.Size([60000])
  split 0 count: 50000
  split 1 count: 10000
